In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import stft, istft
import scipy.ndimage as ndimage

# ==========================================
# 1 Y 2. CARGA DEL AUDIO Y FFT
# ==========================================
samplerate, data = wavfile.read('Prueba.wav')
if len(data.shape) > 1:
    data = data[:, 0]

data_norm = data / np.max(np.abs(data))
f, time_stft, Zxx = stft(data_norm, fs=samplerate, nperseg=2048)
magnitud = np.abs(Zxx)
fase = np.angle(Zxx)

# ==========================================
# 3. PERFIL DE RUIDO
# ==========================================
tiempo_ruido_segundos = 0.5 
muestras_ruido = int(tiempo_ruido_segundos / (time_stft[1] - time_stft[0]))
perfil_ruido = np.mean(magnitud[:, :muestras_ruido], axis=1, keepdims=True)

# ==========================================
# 4. MÁSCARA CALIBRADA (MÁS AGRESIVA)
# ==========================================
print("Aplicando filtro con mayor agresividad contra el ruido ambiente...")

# A) Amplificamos el perfil de ruido para obligar a la fórmula a ser estricta
# Si aún escuchas ambiente, sube este número a 4.0 o 5.0. Si tu voz se empieza a cortar, bájalo a 2.0.
AGRESIVIDAD = 3.5 
ruido_amplificado = perfil_ruido * AGRESIVIDAD

snr = magnitud / (ruido_amplificado + 1e-10)
mascara = 1.0 - (1.0 / snr)

# B) Destruimos el "piso de ruido". Antes dejábamos 0.05 (5%), ahora dejamos casi nada (0.001)
mascara[mascara < 0.001] = 0.001 

# C) Mantenemos el suavizado para evitar que regresen las burbujas
mascara_suavizada = ndimage.gaussian_filter(mascara, sigma=(1, 2))
magnitud_limpia = magnitud * mascara_suavizada

# ==========================================
# 5. RECONSTRUCCIÓN Y EXPORTACIÓN
# ==========================================
Zxx_limpio = magnitud_limpia * np.exp(1j * fase)
_, audio_recuperado = istft(Zxx_limpio, fs=samplerate)

audio_final = np.int16(audio_recuperado * 32767 / np.max(np.abs(audio_recuperado)))
wavfile.write('Prueba_Sustraccion_Calibrada.wav', samplerate, audio_final)

print("¡Listo! Escucha el archivo 'Prueba_Sustraccion_Calibrada.wav'")

Aplicando filtro con mayor agresividad contra el ruido ambiente...
¡Listo! Escucha el archivo 'Prueba_Sustraccion_Calibrada.wav'


/var/folders/jd/bjhpr0zj7nqbr4sfnbqjr33m0000gn/T/ipykernel_80888/1358623070.py:37: RuntimeWarning: divide by zero encountered in divide
  mascara = 1.0 - (1.0 / snr)
